# embedding.py — Interactive Test

Testa `OpenAIEmbeddingProvider`, `LocalEmbeddingProvider` e `embed_chunks_in_batches`.

Os testes da seção **Local** rodam sem nenhuma chave de API.
Os testes da seção **OpenAI** requerem a variável de ambiente `OPENAI_API_KEY` ou a célula de configuração abaixo.

```bash
pip install -r requirements.txt
```

In [ ]:
import sys
import os
from pathlib import Path

ROOT = Path(".").resolve().parent
PDF_DIR = ROOT / "pdf_documents"
sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("PDF directory:", PDF_DIR)

In [ ]:
from core.embedding import (
    get_embedding_provider,
    embed_chunks_in_batches,
    BATCH_SIZE,
)
from core.chunking import Chunk

print("Import OK")
print(f"Default BATCH_SIZE={BATCH_SIZE}")

## Configuração da chave OpenAI (opcional)

Preencha apenas se quiser rodar as seções OpenAI. Deixe vazio para pular.

In [ ]:
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")  # ou cole sua chave aqui
HAS_OPENAI = bool(OPENAI_API_KEY)
print("OpenAI key disponível:", HAS_OPENAI)

## Helper

In [ ]:
def print_embeddings_summary(embeddings: list[list[float]], label: str = "") -> None:
    if label:
        print(f"=== {label} ===")
    print(f"Quantidade de vetores : {len(embeddings)}")
    if embeddings:
        dims = set(len(e) for e in embeddings)
        print(f"Dimensões únicas      : {dims}")
        print(f"Primeiro vetor ([:5]) : {embeddings[0][:5]}")
    print()


def make_chunks(n: int, filename: str = "test.pdf") -> list[Chunk]:
    return [
        Chunk(
            text=f"Este é o trecho número {i} do documento de teste para embeddings.",
            filename=filename,
            page_number=(i % 5) + 1,
            chunk_index=i,
        )
        for i in range(n)
    ]

## 1 — get_embedding_provider: seleção correta de provider

In [ ]:
local_provider = get_embedding_provider(None)
assert isinstance(local_provider, LocalEmbeddingProvider), "Sem chave → deve retornar LocalEmbeddingProvider"
print(f"get_embedding_provider(None) → {type(local_provider).__name__} (dims={local_provider.dimensions})")

if HAS_OPENAI:
    openai_provider = get_embedding_provider(OPENAI_API_KEY)
    assert isinstance(openai_provider, OpenAIEmbeddingProvider), "Com chave → deve retornar OpenAIEmbeddingProvider"
    print(f"get_embedding_provider(key) → {type(openai_provider).__name__} (dims={openai_provider.dimensions})")
else:
    print("(OpenAI pulado — sem chave)")

print("OK")

## 2 — LocalEmbeddingProvider: embed básico

In [ ]:
import asyncio

provider = LocalEmbeddingProvider()
texts = [
    "Motor elétrico trifásico de indução.",
    "Especificações técnicas de corrente e tensão.",
    "Procedimento de instalação e manutenção.",
]

embeddings = asyncio.run(provider.embed(texts))

print_embeddings_summary(embeddings, "LocalEmbeddingProvider — 3 textos")

assert len(embeddings) == 3, "Deve retornar 1 vetor por texto"
assert all(len(e) == 384 for e in embeddings), "LocalProvider deve retornar vetores de 384 dims"
assert provider.dimensions == 384
print("OK — 3 vetores de 384 dimensões")

## 3 — LocalEmbeddingProvider: texto vazio é ignorado

In [ ]:
# sentence-transformers aceita strings vazias sem erro, mas o comportamento
# deve ser consistente — verificamos que não lança exceção
texts_with_empty = ["Texto válido.", "", "   ", "Outro texto válido."]
try:
    result = asyncio.run(provider.embed(texts_with_empty))
    print(f"OK — embed com strings vazias não lançou exceção. Vetores retornados: {len(result)}")
except Exception as e:
    print(f"Exceção (esperada se filtro estiver em LocalProvider): {e}")

## 4 — LocalEmbeddingProvider: similaridade semântica básica
Dois textos semanticamente similares devem ter cosine similarity maior que textos não relacionados.

In [ ]:
import math

def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x**2 for x in a))
    norm_b = math.sqrt(sum(x**2 for x in b))
    return dot / (norm_a * norm_b)


sentences = [
    "O motor consome 2.3 kW de energia elétrica.",       # 0 — referência
    "A potência do motor é de 2.3 quilowatts.",          # 1 — similar à 0
    "A receita leva farinha, ovos e manteiga.",           # 2 — não relacionada
]

embs = asyncio.run(provider.embed(sentences))

sim_related   = cosine_similarity(embs[0], embs[1])
sim_unrelated = cosine_similarity(embs[0], embs[2])

print(f"Similaridade (motor ↔ potência)  : {sim_related:.4f}")
print(f"Similaridade (motor ↔ receita)   : {sim_unrelated:.4f}")

assert sim_related > sim_unrelated, "Textos relacionados devem ter similaridade maior"
print("OK — similaridade semântica funciona corretamente")

## 5 — embed_chunks_in_batches: batching com LocalProvider

In [ ]:
chunks = make_chunks(25)

embeddings = asyncio.run(embed_chunks_in_batches(chunks, provider, batch_size=10))

print_embeddings_summary(embeddings, f"embed_chunks_in_batches — {len(chunks)} chunks, batch_size=10")

assert len(embeddings) == len(chunks), "Deve retornar 1 vetor por chunk"
assert all(len(e) == 384 for e in embeddings)
print(f"OK — {len(chunks)} chunks → {len(embeddings)} vetores")

## 6 — embed_chunks_in_batches: batch único quando n < batch_size

In [ ]:
chunks_small = make_chunks(5)
embeddings_small = asyncio.run(embed_chunks_in_batches(chunks_small, provider, batch_size=100))

assert len(embeddings_small) == 5
print(f"OK — 5 chunks com batch_size=100 → 1 batch único, {len(embeddings_small)} vetores")

## 7 — Consistência de dimensão: vetores local vs OpenAI são incompatíveis
Confirma que os dois providers produzem dimensões diferentes e que isso seria detectável antes de misturá-los no vector store.

In [ ]:
local = LocalEmbeddingProvider()
print(f"LocalEmbeddingProvider.dimensions  = {local.dimensions}")

if HAS_OPENAI:
    openai = OpenAIEmbeddingProvider(api_key=OPENAI_API_KEY)
    print(f"OpenAIEmbeddingProvider.dimensions = {openai.dimensions}")
    assert local.dimensions != openai.dimensions, "Providers diferentes devem ter dimensões diferentes"
    print("OK — dimensões são incompatíveis entre providers (como esperado)")
else:
    assert local.dimensions == 384
    print("OK — LocalEmbeddingProvider.dimensions == 384")
    print("(verificação OpenAI pulada — sem chave)")

## 8 — OpenAIEmbeddingProvider: embed básico
> Requer `OPENAI_API_KEY`.

In [ ]:
if not HAS_OPENAI:
    print("PULADO — defina OPENAI_API_KEY para rodar este teste")
else:
    oai_provider = OpenAIEmbeddingProvider(api_key=OPENAI_API_KEY)
    texts = [
        "Motor elétrico trifásico de indução.",
        "Especificações técnicas de corrente e tensão.",
    ]

    oai_embeddings = asyncio.run(oai_provider.embed(texts))

    print_embeddings_summary(oai_embeddings, "OpenAIEmbeddingProvider — 2 textos")

    assert len(oai_embeddings) == 2
    assert all(len(e) == 1536 for e in oai_embeddings), "OpenAI deve retornar vetores de 1536 dims"
    print("OK — 2 vetores de 1536 dimensões")

## 9 — OpenAIEmbeddingProvider: textos vazios são filtrados
> Requer `OPENAI_API_KEY`.

In [ ]:
if not HAS_OPENAI:
    print("PULADO — defina OPENAI_API_KEY para rodar este teste")
else:
    oai_provider = OpenAIEmbeddingProvider(api_key=OPENAI_API_KEY)
    texts_with_empty = ["Texto válido.", "", "   ", "Outro texto válido."]
    result = asyncio.run(oai_provider.embed(texts_with_empty))

    # embed() retorna 1 vetor por entrada; vazias recebem vetor zero
    assert len(result) == 4
    print(f"OK — {len(texts_with_empty)} entradas → {len(result)} vetores (vazias recebem vetor zero)")

## 10 — embed_chunks_in_batches com OpenAI: batching real
> Requer `OPENAI_API_KEY`. Envia 15 chunks em batches de 5 (3 chamadas à API).

In [ ]:
if not HAS_OPENAI:
    print("PULADO — defina OPENAI_API_KEY para rodar este teste")
else:
    import time

    oai_provider = OpenAIEmbeddingProvider(api_key=OPENAI_API_KEY)
    chunks = make_chunks(15)

    t0 = time.perf_counter()
    embeddings = asyncio.run(embed_chunks_in_batches(chunks, oai_provider, batch_size=5))
    elapsed = (time.perf_counter() - t0) * 1000

    print_embeddings_summary(embeddings, f"OpenAI embed_chunks_in_batches — {len(chunks)} chunks, batch_size=5")
    print(f"Tempo total: {elapsed:.0f} ms")

    assert len(embeddings) == 15
    assert all(len(e) == 1536 for e in embeddings)
    print("OK — 15 chunks → 15 vetores de 1536 dims em 3 batches")

## 11 — End-to-end com PDFs reais (LocalProvider)
Extrai, chunka e gera embeddings para os PDFs da pasta `pdf_documents`.

In [ ]:
from core.extraction import extract_pdf
from core.chunking import chunk_pages
import time

pdfs = sorted(PDF_DIR.glob("*.pdf"))
print(f"{len(pdfs)} PDFs encontrados:")
for p in pdfs:
    print(" ", p.name)

In [ ]:
local_provider = LocalEmbeddingProvider()

for pdf_path in pdfs:
    file_bytes = pdf_path.read_bytes()
    result = extract_pdf(file_bytes, pdf_path.name)

    if not result.has_extractable_text:
        print(f"{pdf_path.name}: sem texto extraível — pulando")
        continue

    chunks = chunk_pages(result.pages)

    t0 = time.perf_counter()
    embeddings = asyncio.run(embed_chunks_in_batches(chunks, local_provider))
    elapsed = (time.perf_counter() - t0) * 1000

    dims = set(len(e) for e in embeddings)

    print(f"{pdf_path.name}")
    print(f"  Chunks          : {len(chunks)}")
    print(f"  Vetores gerados : {len(embeddings)}")
    print(f"  Dimensões       : {dims}")
    print(f"  Tempo embedding : {elapsed:.0f} ms")
    print()

    assert len(embeddings) == len(chunks), "1 vetor por chunk"
    assert dims == {384}, "Todos vetores devem ter 384 dims (LocalProvider)"

print("OK — pipeline completo para todos os PDFs")